# Does a self-aware walk make a difference?

This notebook builds on the three benchmark notebooks. A decision model returns a probability for every option, but the plain walk only follows its choice. A
**self-aware walk** also uses how sure the model was:

- **abstain**: give no answer when the walk's confidence is low;
- **pick**: when the walk ends on several entities, ask the model which one answers the question, and keep it
  alone if the model is sure;
- **escalate**: send an unsure step to a stronger model, and follow that model's choice instead.

Escalating more always costs more, abstaining more always leaves fewer, more accurate answers, and cutting an answer
always changes its score, so better numbers alone prove nothing. The probabilities make a difference only if the
self-aware walk beats **doing the same thing at random**: abstaining on the same share of questions, cutting the same
answers, or escalating the same share of steps. Every test below makes that comparison.

You need:

- a run of the decision model from `2_run_benchmark.ipynb`, for checks 1 and 2 and for abstention (no new calls);
- `OPENROUTER_API_KEY` in `.env` as well, for picking the answer (one call per walk, no graph needed);
- the questions' graphs in Neo4j (from `1_load_benchmark.ipynb`) too, for escalation.

## Settings

- `RUN_NAME`: a run from `2_run_benchmark.ipynb` that includes `DECISION_MODEL`. The picks and the trees go into the
  same run, so they use the same questions.
- `STRONG_MODEL`: where unsure steps are sent. It only helps where it is right and the decision model is wrong;
  check 3 measures that.
- `MAX_ESCALATED`: the main test's rule, fixed before you look at any result: the threshold may send at most this
  share of steps to the strong model.
- `BUDGET_USD`: picking answers or growing trees stops once this much is spent. Rerun the cell to continue.

In [ ]:
BENCHMARK = "webqsp"
RUN_NAME = "run-001"
DECISION_MODEL = "liquid/d1-ultra"
STRONG_MODEL = "openai/gpt-5.6-luna"
MAX_ESCALATED = 0.20
BUDGET_USD = 2.00

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

import json

import pandas as pd

from utils import BENCHMARKS, DecisionModel, LLM, load_decisions, load_results, open_run
from utils import self_aware as sa
from utils.settings import DATA

dm, strong = DecisionModel(DECISION_MODEL), LLM(STRONG_MODEL)
results = load_results(BENCHMARK, [RUN_NAME])
results = results[results["model"] == dm.label]
decisions = load_decisions(RUN_NAME, BENCHMARK, dm.label)
signals = sa.walk_signals(results, decisions)
print(f"{len(signals)} walks of {dm.label} in {RUN_NAME}")

## Check 1: is there anything to put a threshold on?

A threshold can only act on the steps below 1.00. If most steps sit at 1.00, the wrong answers should mostly come from
walks with a step below 1.00, or the probabilities leave a threshold little to work with.

In [ ]:
print(sa.resolution(decisions).round(3).to_string(), "\n")
sure = signals["confidence"].ge(1).map({True: "every step at 1.00, or no model step", False: "a step below 1.00"})
signals.groupby(sure)["exact"].agg(walks="size", exactly_correct="mean")

## Check 2: do the probabilities know when the walk is wrong?

AUROC measures how well a signal ranks the walks that were exactly right above the ones that were not: 0.5 is
chance, 1 is perfect. The confidence is compared with signals that every walk gives for free. If it is not better
than those, the probabilities add nothing. The intervals come from resampling the questions.

In [ ]:
sa.signal_table(signals)

## Abstention: answer only when sure

Keep the most confident answers and drop the rest. The curve shows the error of the answers kept, as fewer are kept.
Abstaining at random gives a flat line at the overall error, and an oracle that drops exactly the wrong answers gives
the best curve possible. **gap closed** is how far the confidence gets from random towards the oracle: 0 is no
better than random, 1 is as good as the oracle.

In [ ]:
curves = pd.DataFrame({"by confidence": sa.risk_coverage(signals["confidence"], signals["score"]),
                       "oracle": sa.risk_coverage(signals["score"], signals["score"])})
curves["at random"] = 1 - signals["score"].mean()
curves.plot(xlabel="share of answers kept", ylabel="error of the answers kept", grid=True)
sa.abstention(signals)

Calibration: the walks grouped by confidence, with how often each group was exactly right. Abstaining on one
benchmark only needs the ranking above. A threshold that works on every benchmark also needs the two columns to
match.

In [ ]:
table, ece = sa.calibration(signals)
print(f"Expected calibration error: {ece:.3f}")
table

## Picking the answer

A step follows a relation to all of its targets, so a walk often ends on several entities when only some of them
answer the question. The **pick** asks the decision model once more, after the walk, which of those entities answers
it, with the same path the walk showed it. If the model gives one entity a probability of at least `sa.SURE` (0.9),
the answer becomes that entity alone; otherwise the walk's answer stays as it was. That is one call per walk that
ended on two or more entities, about a cent per 1,000 questions with d1-ultra, and no graph is needed.

Cutting an answer always changes its score, so here too the test is against doing the same thing **at random**:
keeping one entity at random wherever the pick kept one, scored exactly as an expectation. The **oracle** keeps
exactly the entities that are gold answers: the most that any pick could gain.

In [ ]:
await sa.pick_answers(RUN_NAME, BENCHMARK, dm, budget_usd=BUDGET_USD)
picked = sa.pick_outcomes(RUN_NAME, BENCHMARK, dm.label)
print(f"Asked after {picked['asked'].sum()} of {len(picked)} walks (those that ended on 2 to 255 entities); "
      f"kept one entity alone in {picked['cut'].sum()} of them")
ways = ["walk", "picked", "at random", "oracle"]
pd.DataFrame({"score": picked[ways].mean().values, "exactly right": picked[[w + " exact" for w in ways]].mean().values},
             index=ways).round(4)

The pick makes a difference if it beats both the walk's own answer and cutting at random, with intervals above zero.
`SURE` was chosen on d1-ultra's WebQSP test walks and then tested with no tuning: on 1,000 fresh WebQSP training
questions the pick raised F1 by 1.6 points and exact accuracy by 2.7, and on 1,000 CWQ questions it raised P@1 by
3.6 and exact accuracy by 5.4. Cutting at random did not raise the main score on either benchmark.

In [ ]:
pd.DataFrame({
    "score: picked minus walk": sa.paired_test(picked["picked"], picked["walk"]),
    "exactly right: picked minus walk": sa.paired_test(picked["picked exact"], picked["walk exact"]),
    "score: picked minus at random": sa.paired_test(picked["picked"], picked["at random"]),
    "exactly right: picked minus at random": sa.paired_test(picked["picked exact"], picked["at random exact"]),
}).T.round(4)

In [ ]:
sa.plot_picks({BENCHMARK: picked}, title=f"What the pick added to {RUN_NAME}, in points (score: {BENCHMARKS[BENCHMARK].metric})")

## Escalation: disagreement trees

A walk that escalates costs more than one that does not, so comparing their scores alone is unfair. Instead, every
question gets a **disagreement tree**: at every step both models choose. Where they agree the walk goes on once, and
where they disagree it follows both choices. With at most 4 steps a tree has at most 15 decision points, and usually
little more than one path, because the models mostly agree.

A tree holds every walk that any escalation policy could make on its question. So every policy (each threshold,
random escalation, each model alone) is scored on the same trees, with no new calls and no run-to-run noise between
policies. Random escalation is scored exactly, as an expectation.

Start small: load the 200-question sample in `1_load_benchmark.ipynb` (`SOURCE = "sample"`, `SIZE = 200`) and run it
in `2_run_benchmark.ipynb` first. With GPT-5.6 Luna, the trees for the samples of all three benchmarks cost about $1.

In [ ]:
questions = json.loads((DATA / "loaded" / f"{BENCHMARK}.json").read_text())["questions"]
run = open_run(BENCHMARK, questions, name=RUN_NAME)  # refused if the run holds other questions
await sa.grow_trees(run, dm, strong, budget_usd=BUDGET_USD)
trees = sa.load_trees(RUN_NAME, BENCHMARK, dm.label, strong.label)
print(f"{len(trees)} trees")

### Check 3: is there room to improve?

On each question, the better of the two models' answers is the most that any router between them can reach. If that
is barely above the decision model alone, escalation cannot help, however good the trigger. A strong model that is no
more accurate overall can still help, if its mistakes fall on other questions.

In [ ]:
alone = {"decision model": sa.outcomes(trees, sa.threshold_policy(0)),
         "strong model": sa.outcomes(trees, sa.threshold_policy(1.01))}
sa.headroom(alone["decision model"], alone["strong model"])

### Escalating by confidence against escalating at random

Each point of the threshold line escalates the steps where the decision model was less sure than the threshold. The
random line escalates exactly the same shares of steps, picked at random. The probabilities make a difference only
where the threshold line is above the random line. At a share of 0, both are the decision model alone. At a share of
1, both are the strong model's walk, paying for both models.

In [ ]:
table = sa.sweep(trees)
sa.plot_sweep(table)
table[table["policy"] != "random"].drop_duplicates(subset=["escalated steps", "score"]).round(3)

### The main test

The rule is fixed in the settings: the highest threshold that escalates at most `MAX_ESCALATED` of the steps. So that
the threshold is never tuned on the questions it is tested on, the questions are split into two halves. The threshold
is picked on one half and tested on the other, then the halves swap. Each half is also scored with random escalation
of exactly the share of steps that the threshold escalated there. The comparison is paired: the same questions, side
by side.

On several benchmarks, adjust the main test's p-values together, for example
`sa.holm({"metaqa": 0.01, "webqsp": 0.03, "cwq": 0.2})`, and compare each with 0.05.

In [ ]:
per_question, picks = sa.cross_fit(trees, MAX_ESCALATED)
display(picks.round(3))
pd.DataFrame({
    "threshold against random escalation": sa.paired_test(per_question["score"], per_question["score_random"]),
    "threshold against the decision model alone": sa.paired_test(per_question["score"], alone["decision model"]["score"]),
}).T

## Does it converge?

Three things should settle.

**1. Each walk ends properly.** A self-aware walk should not run into the step limit more often, or walk back to
entities it already stood on ("looped").

In [ ]:
pd.DataFrame({name: frame[["step_limit", "looped"]].mean()
              for name, frame in {**alone, "threshold (main test)": per_question}.items()}).round(3)

**2. The threshold settles.** Both halves should pick similar thresholds, and the share of steps escalated should
hold on the half it was tested on (the first table of the main test). Picking the threshold on more questions should
make the gain level off.

In [ ]:
rows = []
for size in (25, 50, 100, 200, 400, 800):
    if size > len(trees) // 2:
        break
    scored, picked = sa.cross_fit(trees, MAX_ESCALATED, tune_size=size)
    rows.append({"tuning questions": size, "thresholds picked": list(picked["threshold"]),
                 "gain over random": (scored["score"] - scored["score_random"]).mean()})
pd.DataFrame(rows).round(4)

**3. The measured gain settles.** Its interval shrinks as you add questions, and it must be bigger than the noise
between two identical runs. To measure that noise, run the decision model again in `2_run_benchmark.ipynb` on the same
questions, under a new run name, and set `REPEAT_RUN`.

In [ ]:
REPEAT_RUN = None  # for example "run-002": the same decision model on the same questions, run again
if REPEAT_RUN:
    again = load_results(BENCHMARK, [REPEAT_RUN])
    again = again[(again["model"] == dm.label) & again["score"].notna()].set_index("qid")
    print("Second run minus first run:", sa.paired_test(again["score"], signals["score"]))
    print(f"Exactly right in one run but not the other: {(again['exact'] != signals['exact'].reindex(again.index)).mean():.1%}")

Finally, run the chosen threshold for real. It asks the strong model only where the decision model is unsure, so it
costs much less than the trees. Its result should match what the trees predicted, within the interval.

Compare the share of escalated steps, not the dollars. The strong model has just seen these prompts while the trees
grew, so prompt caching can make the real run's strong calls far cheaper than they would be cold: with GPT-5.6 Luna,
about a tenth of the cost on WebQSP and CWQ.

In [ ]:
THRESHOLD = float(picks["threshold"].min())  # or the threshold you settled on
await sa.grow_trees(run, dm, strong, threshold=THRESHOLD, budget_usd=BUDGET_USD)
real = sa.outcomes(sa.load_trees(RUN_NAME, BENCHMARK, dm.label, strong.label, THRESHOLD), sa.threshold_policy(THRESHOLD))
predicted = sa.outcomes(trees, sa.threshold_policy(THRESHOLD))
common = real.index.intersection(predicted.index)
print("Real run minus prediction:", sa.paired_test(real["score"], predicted["score"]))
pd.DataFrame({"real run": sa.policy_summary(real.loc[common]),
              "predicted by the trees": sa.policy_summary(predicted.loc[common])}).round(4)

## How many questions you need

The spread of the per-question differences in this run gives how many questions a paired test needs to detect each
gain, with 80% power and 0.05 split over three benchmarks. If that is more than you ran, the main test can miss a
real gain of that size.

In [ ]:
spread = (per_question["score"] - per_question["score_random"]).var()
pd.Series({f"a gain of {gain:.0%}": sa.questions_needed(spread, gain) for gain in (0.01, 0.02, 0.03, 0.05)},
          name="questions needed")